In [2]:
import os
import requests
import asyncpg
import pandas as pd
import nest_asyncio

nest_asyncio.apply()

pd.set_option("display.max_colwidth", 500)


## 1. Cấu hình

Sửa `DB_URL` cho đúng database của bạn. `EMBED_URL` và `EMBED_MODEL` phải giống notebook dùng để embed dữ liệu vào database.


In [3]:
ROOT = Path.cwd()

META_PATH = ROOT / "knowledge" / "metadata.md"
BCTC_PATH = ROOT / "knowledge" / "bctc_description.md"

from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
EMBED_MODEL = "text-embedding-3-small"
client = OpenAI(api_key=OPENAI_API_KEY)

# số object mỗi chunk (tuning ở đây)
BCTC_CHUNK_SIZE = 1

## 2. Hàm embed query

Hàm này gọi local embedding server, cùng endpoint/model với lúc insert dữ liệu.


In [4]:
def embed_query(text: str):
    res = client.embeddings.create(
        model=EMBED_MODEL,
        input=[text],
        dimensions=1024
    )
    return res.data[0].embedding

## 3. Test nhanh embedding

Cell này kiểm tra local embedding server có chạy không và dimension vector là bao nhiêu.


In [5]:
test_emb = embed_query("doanh thu thuần là gì")
print("Embedding dimension:", len(test_emb))
print("First 5 values:", test_emb[:5])


Embedding dimension: 1024
First 5 values: [-0.032458677887916565, -0.004709488246589899, -0.004026289097964764, 0.0006242543458938599, -0.029546014964580536]


## 4. Hàm vector search

`doc_type` có thể để `None` để search toàn bộ, hoặc truyền ví dụ `"bctc_description"` để chỉ search trong BCTC description.


In [6]:
async def vector_search(
    query: str,
    top_k: int = 5,
    doc_type: str | None = None
) -> pd.DataFrame:
    columns = [
        "id",
        "doc_type",
        "source_file",
        "chunk_id",
        "chunk_index",
        "content",
        "similarity",
    ]

    top_k = max(1, min(int(top_k), 100))

    emb = embed_query(query)
    emb_str = "[" + ",".join(map(str, emb)) + "]"

    conn = await asyncpg.connect(DB_URL)

    try:
        if doc_type is not None and str(doc_type).strip() != "":
            rows = await conn.fetch(
                f"""
                SELECT
                    id,
                    doc_type,
                    source_file,
                    chunk_id,
                    chunk_index,
                    content,
                    1 - (embedding <=> $1::vector) AS similarity
                FROM {TABLE_NAME}
                WHERE doc_type = $2
                ORDER BY embedding <=> $1::vector
                LIMIT $3
                """,
                emb_str,
                doc_type,
                top_k,
            )
        else:
            rows = await conn.fetch(
                f"""
                SELECT
                    id,
                    doc_type,
                    source_file,
                    chunk_id,
                    chunk_index,
                    content,
                    1 - (embedding <=> $1::vector) AS similarity
                FROM {TABLE_NAME}
                ORDER BY embedding <=> $1::vector
                LIMIT $2
                """,
                emb_str,
                top_k,
            )

        return pd.DataFrame([dict(r) for r in rows], columns=columns)

    finally:
        await conn.close()

## 5. Test search trong `bctc_description`


In [7]:
df = await vector_search(
    "chi phí quản lý doanh nghiệp (QLDN)",
    top_k=5,
    doc_type="bctc"
)

df[["doc_type", "source_file", "similarity", "content"]]


,doc_type,source_file,similarity,content
0,bctc,bctc_description.md,0.706861,"[\n {\n ""raw_name"": ""Chi phí quản lý DN"",\n ""old_ind_code"": ""cp_qldn"",\n ""norm_name"": ""Chi phí quản lý doanh nghiệp""\n }\n]"
1,bctc,bctc_description.md,0.706861,"[\n {\n ""raw_name"": ""Chi phí quản lý DN"",\n ""old_ind_code"": ""cp_qldn"",\n ""norm_name"": ""Chi phí quản lý doanh nghiệp""\n }\n]"
2,bctc,bctc_description.md,0.696248,"[\n {\n ""raw_name"": ""Chi phí quản lý doanh nghiệp"",\n ""old_ind_code"": ""cp_qldn"",\n ""norm_name"": ""Chi phí quản lý doanh nghiệp""\n }\n]"
3,bctc,bctc_description.md,0.696248,"[\n {\n ""raw_name"": ""Chi phí quản lý doanh nghiệp"",\n ""old_ind_code"": ""cp_qldn"",\n ""norm_name"": ""Chi phí quản lý doanh nghiệp""\n }\n]"
4,bctc,bctc_description.md,0.696248,"[\n {\n ""raw_name"": ""Chi phí quản lý doanh nghiệp"",\n ""old_ind_code"": ""cp_qldn"",\n ""norm_name"": ""Chi phí quản lý doanh nghiệp""\n }\n]"


In [8]:
df = await vector_search(
    "ROE, ROA, Pb, PE",
    top_k=5,
    doc_type="metadata"
)

df[["doc_type", "source_file", "similarity", "content"]]


,doc_type,source_file,similarity,content
0,metadata,metadata.md,0.414837,## 9) Bảng financial_ratio\n\nMục đích: Lưu các tỷ số tài chính chuẩn hóa theo quý/năm cho từng mã.\n\n| Tên cột | Kiểu dữ liệu | Mô tả tiếng Việt |\n|---|---|---|\n| id | bigserial | ID tự tăng nội bộ. |\n| ind_code | text | Mã chỉ tiêu nguồn (nếu có). |\n| ticker | varchar(20) | Mã cổ phiếu. |\n| year | integer | Năm tài chính. |\n| quarter | integer | Quý tài chính. |\n| fixed_asset_to_equity | double precision | Tài sản cố định/Vốn chủ sở hữu. |\n| equity_to_charter_capital | double prec...
1,metadata,metadata.md,0.398287,"## 10) Bảng electric_board\n\nMục đích: Dữ liệu chốt bảng điện theo ngày, dùng cho trang price-board và phân tích giao dịch.\n\n| Tên cột | Kiểu dữ liệu | Mô tả tiếng Việt |\n|---|---|---|\n| id | serial | ID nội bộ. |\n| ticker | varchar(20) | Mã cổ phiếu. |\n| exchange | varchar(10) | Sàn giao dịch. |\n| trading_date | date | Ngày giao dịch. |\n| ref_price | numeric(18,2) | Giá tham chiếu. |\n| match_price | numeric(18,2) | Giá khớp. |\n| accumulated_volume | bigint | Khối lượng khớp lũy k..."
2,metadata,metadata.md,0.388259,## 5) Bảng company_overview\n\nMục đích: Lưu hồ sơ doanh nghiệp và phân ngành phục vụ lọc theo ngành.\n\n| Tên cột | Kiểu dữ liệu | Mô tả tiếng Việt |\n|---|---|---|\n| ticker | varchar(10) | Mã cổ phiếu. |\n| overview | text | Mô tả tổng quan doanh nghiệp. |\n| icb_name1 | text | Nhóm ngành cấp 1 (ICB). |\n| icb_name2 | text | Nhóm ngành cấp 2 (ICB). |\n| icb_name3 | text | Nhóm ngành cấp 3 (ICB). |\n| import_time | timestamp | Thời điểm nạp dữ liệu. |\n| exchange | text | Sàn giao dịch (HO...
3,metadata,metadata.md,0.372679,## 8) Bảng macro_economy\n\nMục đích: Dữ liệu vĩ mô theo từng loại tài sản và thời gian.\n\n| Tên cột | Kiểu dữ liệu | Mô tả tiếng Việt |\n|---|---|---|\n| date | date | Ngày dữ liệu vĩ mô. |\n| open | real | Giá trị mở đầu kỳ/ngày. |\n| high | real | Giá trị cao nhất. |\n| low | real | Giá trị thấp nhất. |\n| close | real | Giá trị đóng kỳ/ngày. |\n| volume | bigint | Khối lượng (nếu áp dụng). |\n| asset_type | varchar(20) | Loại tài sản/chỉ số vĩ mô. |
4,metadata,metadata.md,0.371074,"## 6) Bảng bctc\n\nMục đích: Lưu dữ liệu chỉ tiêu báo cáo tài chính chuẩn hóa theo mã, năm, quý và mã chỉ tiêu.\n\n| Tên cột | Kiểu dữ liệu | Mô tả tiếng Việt |\n|---|---|---|\n| ticker | varchar(10) | Mã cổ phiếu. |\n| quarter | varchar(10) | Quý báo cáo (Q1, Q2, Q3, Q4 hoặc dạng tương đương). |\n| year | integer | Năm báo cáo. |\n| ind_name | text | Tên chỉ tiêu gốc (có thể nhiều biến thể). |\n| ind_code | text | Mã chỉ tiêu chuẩn hóa để truy vấn thống nhất. |\n| value | numeric(25,4) | Gi..."
